# Aula 11 — Backward das losses

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/04-deep-learning/m5-redes-neurais-do-zero/notebooks/11-backward-losses-laboratorio.ipynb)

Laboratório reproduzível em **NumPy puro** para derivar e testar MSE, BCE em logits e softmax + cross-entropy. O arquivo versionado não contém outputs; execute as células em ordem.

## Objetivos e ambiente

- Implementar forward e backward com contratos explícitos de shape e redução.
- Confirmar MSE, sigmoid + BCE e softmax + CE por diferenças centrais.
- Comparar rotas compostas e fundidas sem esconder instabilidade com clipping.
- Testar upstream, pesos, máscaras, invariância por deslocamento e eixo de classes.

Dependências mínimas:

- Python >= 3.11
- NumPy >= 1.26
- Matplotlib >= 3.8
- nbformat >= 5.9 apenas para validar o arquivo

Não há downloads, credenciais, frameworks de deep learning nem aleatoriedade sem seed.

In [ ]:
from importlib.metadata import version
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260911
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=9, suppress=True)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("nbformat:", version("nbformat"))
print("Seed:", SEED)

## 1. Contratos e reduções

As funções abaixo recusam broadcasting implícito entre predição e alvo. `mean` divide pelo número de elementos na MSE/BCE e pelo número de exemplos na CE; `none` exige um upstream com o shape da loss não reduzida.

In [ ]:
VALID_REDUCTIONS = {"none", "sum", "mean"}


def finite_array(name, value, rank=None):
    arr = np.asarray(value, dtype=np.float64)
    if rank is not None and arr.ndim != rank:
        raise ValueError(f"{name} deve ter rank {rank}; recebido {arr.shape}")
    if not np.all(np.isfinite(arr)):
        raise ValueError(f"{name} contém valor não finito")
    return arr


def same_shape(a_name, a, b_name, b):
    if a.shape != b.shape:
        raise ValueError(f"{a_name} {a.shape} deve coincidir com {b_name} {b.shape}")


def reduce_values(values, reduction):
    if reduction not in VALID_REDUCTIONS:
        raise ValueError(f"redução inválida: {reduction}")
    if reduction == "none":
        return values.copy()
    if reduction == "sum":
        return float(np.sum(values))
    return float(np.mean(values))


def scalar_upstream(upstream):
    up = np.asarray(upstream, dtype=np.float64)
    if up.shape != () or not np.isfinite(up):
        raise ValueError("upstream de loss escalar deve ser escalar e finito")
    return float(up)

## 2. MSE

Para média global sobre $N$ elementos,

$$L=\frac{1}{N}\sum_r(\hat y_r-y_r)^2,
\qquad \frac{\partial L}{\partial\hat y_r}=\frac{2}{N}(\hat y_r-y_r).$$

In [ ]:
def mse_loss(y_hat, y, reduction="mean"):
    y_hat = finite_array("y_hat", y_hat)
    y = finite_array("y", y)
    same_shape("y_hat", y_hat, "y", y)
    return reduce_values((y_hat - y) ** 2, reduction)


def mse_backward(y_hat, y, reduction="mean", upstream=1.0):
    y_hat = finite_array("y_hat", y_hat)
    y = finite_array("y", y)
    same_shape("y_hat", y_hat, "y", y)
    local = 2.0 * (y_hat - y)
    if reduction == "none":
        up = finite_array("upstream", upstream)
        same_shape("upstream", up, "loss não reduzida", local)
        return local * up
    up = scalar_upstream(upstream)
    if reduction == "sum":
        return local * up
    if reduction == "mean":
        return local * (up / local.size)
    raise ValueError(f"redução inválida: {reduction}")


y_hat = np.array([2.0, 1.0])
y = np.array([1.0, -1.0])
print("MSE mean:", mse_loss(y_hat, y))
print("gradiente:", mse_backward(y_hat, y))

assert np.isclose(mse_loss(y_hat, y), 2.5)
assert np.allclose(mse_backward(y_hat, y), [1.0, 2.0])
assert np.allclose(mse_backward(y_hat, y, "sum"), 2 * (y_hat - y))
assert np.allclose(mse_backward(y_hat, y, "none", [2.0, 3.0]), [4.0, 12.0])

### Dupla média

Duplicar exemplos idênticos reduz pela metade o gradiente de cada cópia, mas a soma que chega a parâmetros compartilhados permanece igual. Dividir novamente no otimizador quebraria essa invariância.

In [ ]:
single_grad = mse_backward(np.array([2.0]), np.array([1.0]), "mean")
duplicated_grad = mse_backward(np.array([2.0, 2.0]), np.array([1.0, 1.0]), "mean")
wrong_double_mean = duplicated_grad / 2

print("gradiente original:", single_grad)
print("gradientes das cópias:", duplicated_grad)
print("soma correta:", duplicated_grad.sum())
print("soma com dupla média:", wrong_double_mean.sum())

assert np.isclose(duplicated_grad.sum(), single_grad.sum())
assert np.isclose(wrong_double_mean.sum(), single_grad.sum() / 2)

## 3. BCE diretamente dos logits

Usamos $\ell(z,y)=\operatorname{softplus}(z)-yz$ e uma sigmoid por ramos para evitar overflow. O backward fundido é $\sigma(z)-y$, antes da escala da redução.

In [ ]:
def sigmoid_stable(z):
    z = finite_array("z", z)
    out = np.empty_like(z)
    positive = z >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    ez = np.exp(z[~positive])
    out[~positive] = ez / (1.0 + ez)
    return out


def bce_logits_loss(z, y, reduction="mean"):
    z = finite_array("z", z)
    y = finite_array("y", y)
    same_shape("z", z, "y", y)
    if np.any((y < 0) | (y > 1)):
        raise ValueError("alvos BCE devem estar em [0, 1]")
    elementwise = np.logaddexp(0.0, z) - y * z
    return reduce_values(elementwise, reduction)


def bce_logits_backward(z, y, reduction="mean", upstream=1.0):
    z = finite_array("z", z)
    y = finite_array("y", y)
    same_shape("z", z, "y", y)
    if np.any((y < 0) | (y > 1)):
        raise ValueError("alvos BCE devem estar em [0, 1]")
    local = sigmoid_stable(z) - y
    if reduction == "none":
        up = finite_array("upstream", upstream)
        same_shape("upstream", up, "loss não reduzida", local)
        return local * up
    up = scalar_upstream(upstream)
    if reduction == "sum":
        return local * up
    if reduction == "mean":
        return local * (up / local.size)
    raise ValueError(f"redução inválida: {reduction}")


z_bin = np.array([0.0, 2.0, -2.0])
y_bin = np.array([0.0, 1.0, 0.0])
p_bin = sigmoid_stable(z_bin)
print("p:", p_bin)
print("BCE mean:", bce_logits_loss(z_bin, y_bin))
print("dL/dz:", bce_logits_backward(z_bin, y_bin))

assert np.allclose(p_bin, [0.5, 0.880797078, 0.119202922], atol=1e-9)
assert np.allclose(bce_logits_backward(z_bin, y_bin), (p_bin - y_bin) / 3)

### Cadeia explícita versus forma fundida

Em valores moderados, $\partial\ell/\partial p=(p-y)/(p(1-p))$ multiplicado por $dp/dz=p(1-p)$ coincide com $p-y$. A forma intermediária não deve ser usada em logits extremos.

In [ ]:
z_moderate = np.array([-3.0, -0.2, 0.7, 4.0])
y_moderate = np.array([0.0, 1.0, 0.0, 1.0])
p = sigmoid_stable(z_moderate)
dell_dp = (p - y_moderate) / (p * (1.0 - p))
dp_dz = p * (1.0 - p)
chained = dell_dp * dp_dz
fused = bce_logits_backward(z_moderate, y_moderate, "sum")

print("erro cadeia × fundida:", np.max(np.abs(chained - fused)))
assert np.allclose(chained, fused, rtol=1e-13, atol=1e-13)

### Logits extremos

`logaddexp` mantém loss e gradiente finitos para $z=\pm1000$. Clipping de probabilidades devolve outra loss e pode zerar o gradiente pela operação de corte.

In [ ]:
z_extreme = np.array([-1000.0, 1000.0, 1000.0, -1000.0])
y_extreme = np.array([0.0, 1.0, 0.0, 1.0])
loss_extreme = bce_logits_loss(z_extreme, y_extreme, "none")
grad_extreme = bce_logits_backward(z_extreme, y_extreme, "sum")

eps_clip = 1e-12
p_extreme = sigmoid_stable(z_extreme)
p_clipped = np.clip(p_extreme, eps_clip, 1 - eps_clip)
loss_clipped = -(y_extreme * np.log(p_clipped) + (1-y_extreme) * np.log1p(-p_clipped))

print("loss estável:", loss_extreme)
print("gradiente estável:", grad_extreme)
print("loss após clipping:", loss_clipped)

assert np.all(np.isfinite(loss_extreme)) and np.all(np.isfinite(grad_extreme))
assert np.allclose(loss_extreme, [0.0, 0.0, 1000.0, 1000.0])
assert loss_clipped[2] < 28 and loss_clipped[3] < 28

## 4. Softmax + cross-entropy em logits

Calculamos $\ell_i=\operatorname{LSE}(z_i)-q_i^\top z_i$ com classes no eixo 1. O backward fundido é $P-Q$; `mean` divide por $m$, não por $mK$.

In [ ]:
def validate_multiclass(z, q):
    z = finite_array("z", z, rank=2)
    q = finite_array("q", q, rank=2)
    same_shape("z", z, "q", q)
    if z.shape[1] < 2:
        raise ValueError("CE requer ao menos duas classes")
    if np.any(q < 0) or not np.allclose(q.sum(axis=1), 1.0, atol=1e-12):
        raise ValueError("cada linha de q deve ser uma distribuição")
    return z, q


def softmax_stable(z):
    z = finite_array("z", z, rank=2)
    shifted = z - np.max(z, axis=1, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / exp_shifted.sum(axis=1, keepdims=True)


def logsumexp_rows(z):
    z = finite_array("z", z, rank=2)
    maximum = np.max(z, axis=1, keepdims=True)
    return (maximum + np.log(np.exp(z - maximum).sum(axis=1, keepdims=True))).ravel()


def ce_logits_loss(z, q, reduction="mean"):
    z, q = validate_multiclass(z, q)
    per_example = logsumexp_rows(z) - np.sum(q * z, axis=1)
    return reduce_values(per_example, reduction)


def ce_logits_backward(z, q, reduction="mean", upstream=1.0):
    z, q = validate_multiclass(z, q)
    local = softmax_stable(z) - q
    m = z.shape[0]
    if reduction == "none":
        up = finite_array("upstream", upstream, rank=1)
        if up.shape != (m,):
            raise ValueError(f"upstream deve ter shape {(m,)}; recebido {up.shape}")
        return local * up[:, None]
    up = scalar_upstream(upstream)
    if reduction == "sum":
        return local * up
    if reduction == "mean":
        return local * (up / m)
    raise ValueError(f"redução inválida: {reduction}")

### Exemplo resolvido e conservação do gradiente

Para logits $[2,1,0]$ e alvo one-hot da classe 0, esperamos loss $0{,}407606$ e gradiente $[-0{,}334759,0{,}244728,0{,}090031]$.

In [ ]:
z_ce = np.array([[2.0, 1.0, 0.0]])
q_ce = np.array([[1.0, 0.0, 0.0]])
p_ce = softmax_stable(z_ce)
loss_ce = ce_logits_loss(z_ce, q_ce)
grad_ce = ce_logits_backward(z_ce, q_ce)

print("probabilidades:", p_ce)
print("CE:", loss_ce)
print("dL/dz:", grad_ce)
print("soma do gradiente:", grad_ce.sum(axis=1))

assert np.allclose(p_ce, [[0.665240956, 0.244728471, 0.090030573]], atol=1e-9)
assert np.isclose(loss_ce, 0.407605964, atol=1e-9)
assert np.allclose(grad_ce.sum(axis=1), 0.0, atol=1e-15)

### Alvo denso, deslocamento e rota pelo Jacobiano

O teste usa label smoothing, confirma invariância ao somar uma constante por linha e reconstrói a VJP por um Jacobiano softmax explícito apenas para auditoria.

In [ ]:
z_dense = np.array([[2.0, 1.0, 0.0], [-0.5, 0.2, 1.4]])
q_dense = np.array([[0.8, 0.1, 0.1], [0.05, 0.15, 0.8]])
p_dense = softmax_stable(z_dense)
grad_fused = ce_logits_backward(z_dense, q_dense, "sum")

def softmax_jacobian(p_row):
    return np.diag(p_row) - np.outer(p_row, p_row)

grad_chained = np.empty_like(grad_fused)
for i in range(z_dense.shape[0]):
    dloss_dp = -q_dense[i] / p_dense[i]
    grad_chained[i] = softmax_jacobian(p_dense[i]).T @ dloss_dp

row_shift = np.array([[1000.0], [-777.0]])
print("erro Jacobiano × fundida:", np.max(np.abs(grad_chained - grad_fused)))
print("erro de loss após deslocamento:", abs(ce_logits_loss(z_dense + row_shift, q_dense, "sum") - ce_logits_loss(z_dense, q_dense, "sum")))

assert np.allclose(grad_chained, grad_fused, atol=2e-15)
assert np.isclose(ce_logits_loss(z_dense + row_shift, q_dense, "sum"), ce_logits_loss(z_dense, q_dense, "sum"), atol=2e-13)
assert np.allclose(ce_logits_backward(z_dense + row_shift, q_dense, "sum"), grad_fused, atol=2e-14)

## 5. Diferenças centrais

O verificador perturba uma coordenada por vez em `float64`. Ele recebe uma função escalar; por isso testaremos cada loss com redução `mean`.

In [ ]:
def numerical_gradient(function, x, epsilon=1e-6):
    x = finite_array("x", x).copy()
    gradient = np.empty_like(x)
    for index in np.ndindex(x.shape):
        old = x[index]
        x[index] = old + epsilon
        plus = float(function(x))
        x[index] = old - epsilon
        minus = float(function(x))
        x[index] = old
        gradient[index] = (plus - minus) / (2 * epsilon)
    return gradient


def symmetric_relative_error(a, b):
    numerator = np.linalg.norm(a - b)
    denominator = max(1.0, np.linalg.norm(a), np.linalg.norm(b))
    return float(numerator / denominator)


yhat_gc = rng.normal(size=(3, 2))
y_gc = rng.normal(size=(3, 2))
z_bce_gc = rng.normal(size=(4, 2))
y_bce_gc = rng.integers(0, 2, size=(4, 2)).astype(float)
z_ce_gc = rng.normal(size=(3, 4))
labels_gc = np.array([0, 2, 1])
q_ce_gc = np.eye(4)[labels_gc]

checks = {
    "MSE": symmetric_relative_error(
        mse_backward(yhat_gc, y_gc),
        numerical_gradient(lambda value: mse_loss(value, y_gc), yhat_gc),
    ),
    "BCE logits": symmetric_relative_error(
        bce_logits_backward(z_bce_gc, y_bce_gc),
        numerical_gradient(lambda value: bce_logits_loss(value, y_bce_gc), z_bce_gc),
    ),
    "CE logits": symmetric_relative_error(
        ce_logits_backward(z_ce_gc, q_ce_gc),
        numerical_gradient(lambda value: ce_logits_loss(value, q_ce_gc), z_ce_gc),
    ),
}

for name, error in checks.items():
    print(f"{name}: {error:.3e}")
    assert error < 2e-9

## 6. Upstream escalar e teste direcional

Se $J=aL$, o gradiente deve ser $a\nabla L$. O produto interno com uma direção $V$ também deve coincidir com a derivada direcional numérica.

In [ ]:
upstream = 0.37
scaled = ce_logits_backward(z_ce_gc, q_ce_gc, "mean", upstream)
assert np.allclose(scaled, upstream * ce_logits_backward(z_ce_gc, q_ce_gc))

direction = rng.normal(size=z_ce_gc.shape)
direction /= np.linalg.norm(direction)
epsilon = 1e-6
directional_numeric = (
    ce_logits_loss(z_ce_gc + epsilon * direction, q_ce_gc)
    - ce_logits_loss(z_ce_gc - epsilon * direction, q_ce_gc)
) / (2 * epsilon)
directional_analytic = np.sum(ce_logits_backward(z_ce_gc, q_ce_gc) * direction)
directional_error = abs(directional_numeric - directional_analytic)

print("direcional numérica:", directional_numeric)
print("direcional analítica:", directional_analytic)
print("erro:", directional_error)
assert directional_error < 5e-10

## 7. Pesos e máscaras com denominador explícito

Para BCE mascarada, implementamos $L=\sum_iw_i\ell_i/\sum_iw_i$. Posições com peso zero não influenciam nem numerador nem denominador.

In [ ]:
def weighted_bce_logits(z, y, weights):
    z = finite_array("z", z)
    y = finite_array("y", y)
    weights = finite_array("weights", weights)
    same_shape("z", z, "y", y)
    same_shape("z", z, "weights", weights)
    if np.any(weights < 0):
        raise ValueError("pesos devem ser não negativos")
    denominator = float(weights.sum())
    if denominator <= 0:
        raise ValueError("a soma dos pesos deve ser positiva")
    elementwise = np.logaddexp(0.0, z) - y * z
    loss = float(np.sum(weights * elementwise) / denominator)
    gradient = weights * (sigmoid_stable(z) - y) / denominator
    return loss, gradient


z_mask = np.array([0.0, 1.0, -3.0, 4.0])
y_mask = np.array([1.0, 1.0, 0.0, 0.0])
mask = np.array([1.0, 1.0, 0.0, 0.0])
weighted_loss, weighted_grad = weighted_bce_logits(z_mask, y_mask, mask)
wrong_grad = mask * (sigmoid_stable(z_mask) - y_mask) / mask.size

print("loss válida:", weighted_loss)
print("gradiente válido:", weighted_grad)
print("norma válida / norma com denominador bruto:", np.linalg.norm(weighted_grad) / np.linalg.norm(wrong_grad))

assert np.allclose(weighted_grad[2:], 0.0)
assert np.allclose(weighted_grad, 2 * wrong_grad)
try:
    weighted_bce_logits(z_mask, y_mask, np.zeros_like(mask))
except ValueError:
    pass
else:
    raise AssertionError("máscara vazia deveria ser rejeitada")

## 8. Contraprova: eixo de classes incorreto

Uma softmax em `axis=0` normaliza entre exemplos. Além de as linhas não somarem 1, alterar uma amostra muda a “probabilidade” das demais.

In [ ]:
def softmax_wrong_axis0(z):
    shifted = z - np.max(z, axis=0, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=0, keepdims=True)


z_axis = np.array([[2.0, 1.0, 0.0], [0.2, -0.1, 1.3]])
p_right = softmax_stable(z_axis)
p_wrong = softmax_wrong_axis0(z_axis)
mutated = z_axis.copy()
mutated[1] += np.array([50.0, -20.0, 7.0])

right_change = np.max(np.abs(softmax_stable(mutated)[0] - p_right[0]))
wrong_change = np.max(np.abs(softmax_wrong_axis0(mutated)[0] - p_wrong[0]))
print("somas das linhas corretas:", p_right.sum(axis=1))
print("somas das linhas erradas:", p_wrong.sum(axis=1))
print("mudança na linha 0 — eixo correto:", right_change)
print("mudança na linha 0 — eixo errado:", wrong_change)

assert np.allclose(p_right.sum(axis=1), 1.0)
assert not np.allclose(p_wrong.sum(axis=1), 1.0)
assert right_change == 0.0 and wrong_change > 0.1

## 9. Visualização: loss e gradiente binários

O gradiente em logits permanece entre $-1$ e $1$ por elemento. Previsões erradas e confiantes recebem grande magnitude; previsões corretas e confiantes se aproximam de zero.

In [ ]:
grid = np.linspace(-10, 10, 401)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for target, color in [(0.0, "tab:blue"), (1.0, "tab:orange")]:
    targets = np.full_like(grid, target)
    axes[0].plot(grid, bce_logits_loss(grid, targets, "none"), label=f"y={int(target)}", color=color)
    axes[1].plot(grid, bce_logits_backward(grid, targets, "sum"), label=f"y={int(target)}", color=color)
axes[0].set(title="BCE estável", xlabel="logit z", ylabel="loss por elemento")
axes[1].set(title="Backward fundido", xlabel="logit z", ylabel="dℓ/dz")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()
plt.show()

**Texto alternativo do gráfico:** dois painéis em função do logit de −10 a 10. À esquerda, a BCE para alvo 0 cresce aproximadamente de modo linear em logits positivos e, para alvo 1, em logits negativos. À direita, o gradiente para alvo 0 cresce de 0 a 1; para alvo 1, cresce de −1 a 0.

## 10. Auditoria final

Esta célula repete contratos estruturais que devem continuar verdadeiros após qualquer alteração no notebook.

In [ ]:
audit = {
    "MSE resolvida": np.isclose(mse_loss(y_hat, y), 2.5),
    "shape MSE": mse_backward(yhat_gc, y_gc).shape == yhat_gc.shape,
    "BCE finita": np.all(np.isfinite(loss_extreme)),
    "shape BCE": bce_logits_backward(z_bce_gc, y_bce_gc).shape == z_bce_gc.shape,
    "BCE fundida": np.allclose(chained, fused),
    "CE resolvida": np.isclose(loss_ce, 0.407605964, atol=1e-9),
    "shape CE": grad_fused.shape == z_dense.shape,
    "gradiente CE conserva soma": np.allclose(grad_fused.sum(axis=1), 0.0, atol=1e-15),
    "CE invariante a shift": np.isclose(ce_logits_loss(z_dense + row_shift, q_dense, "sum"), ce_logits_loss(z_dense, q_dense, "sum"), atol=2e-13),
    "MSE gradient check": checks["MSE"] < 2e-9,
    "BCE gradient check": checks["BCE logits"] < 2e-9,
    "CE gradient check": checks["CE logits"] < 2e-9,
    "teste direcional": directional_error < 5e-10,
    "máscara zera padding": np.allclose(weighted_grad[2:], 0.0),
    "eixo separa exemplos": right_change == 0.0,
}

for name, passed in audit.items():
    assert passed, name
print(f"Auditoria concluída: {sum(audit.values())}/{len(audit)} grupos aprovados.")

## Conclusões

- MSE média devolve $2(\hat y-y)/N$.
- BCE em logits devolve $(\sigma(z)-y)/N$ sob média elementwise.
- Softmax + CE devolve $(P-Q)/m$ sob média por exemplo.
- A forma fundida é equivalente à cadeia em região moderada e permanece definida em logits extremos.
- Redução, máscara, upstream e eixo são partes do contrato matemático.

Na Aula 12, esses gradientes serão conectados a ativação e camada afim em uma rede escalar completa.